In [23]:
import numpy as np 
import re

import math

In [11]:
docs = [
    "AI is transforming business operations",
    "AI and machine learning improve business efficiency",
    "Operations and analytics drive business growth"
]


In [21]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)   # remove punctuation/numbers
    return text.split()

In [12]:
# for the tokenise 

def tokenizer(doc):
    return doc.lower().split()

In [13]:
# for the BUilt Vocab

def built_vocab(docs):
    vocab = set()
    for doc in docs:
        vocab.update(tokenizer(doc))
    return sorted(list(vocab))



In [14]:
# for the tf

def compute_tf(doc, vocab):
    tokens = tokenizer(doc)
    tf = np.zeros(len(vocab))
    for i, term in enumerate(vocab):
        tf[i] = tokens.count(term) / len(tokens)
    return tf

    

In [15]:
# for idf

def compute_idf(docs, vocab):
    N = len(docs)
    idf = np.zeros(len(vocab))
    for i, term in enumerate(vocab):
        n_t = sum(1 for doc in docs if term in tokenizer(doc))
        idf[i] = math.log((N / (1 + n_t))) + 1  # +1 for smoothing
    return idf


In [16]:
def compute_tfidf(docs):
    vocab = built_vocab(docs)
    idf = compute_idf(docs, vocab)
    tfidf_matrix = []
    for doc in docs:
        tf = compute_tf(doc, vocab)
        tfidf = tf * idf
        tfidf_matrix.append(tfidf)
    return np.array(tfidf_matrix), vocab

In [17]:
tfidf_matrix, vocab = compute_tfidf(docs)


In [ ]:
docs = [
    "AI is transforming business operations",
    "AI and machine learning improve business efficiency",
    "Operations and analytics drive business growth"
]


In [18]:
tfidf_matrix, vocab

(array([[0.2       , 0.        , 0.        , 0.14246359, 0.        ,
         0.        , 0.        , 0.        , 0.28109302, 0.        ,
         0.        , 0.2       , 0.28109302],
        [0.14285714, 0.        , 0.14285714, 0.1017597 , 0.        ,
         0.20078073, 0.        , 0.20078073, 0.        , 0.20078073,
         0.20078073, 0.        , 0.        ],
        [0.        , 0.23424418, 0.16666667, 0.11871965, 0.23424418,
         0.        , 0.23424418, 0.        , 0.        , 0.        ,
         0.        , 0.16666667, 0.        ]]),
 ['ai',
  'analytics',
  'and',
  'business',
  'drive',
  'efficiency',
  'growth',
  'improve',
  'is',
  'learning',
  'machine',
  'operations',
  'transforming'])

In [ ]:
import pandas as pd


df = pd.DataFrame(tfidf_matrix, columns=vocab)

df.index = [f"Document {i+1}" for i in range(len(docs))]

print(df.round(3))  # round values for readability


               ai  analytics    and  business  drive  efficiency  growth  \
Document 1  0.200      0.000  0.000     0.142  0.000       0.000   0.000   
Document 2  0.143      0.000  0.143     0.102  0.000       0.201   0.000   
Document 3  0.000      0.234  0.167     0.119  0.234       0.000   0.234   

            improve     is  learning  machine  operations  transforming  
Document 1    0.000  0.281     0.000    0.000       0.200         0.281  
Document 2    0.201  0.000     0.201    0.201       0.000         0.000  
Document 3    0.000  0.000     0.000    0.000       0.167         0.000  


In [37]:
from numpy.linalg import norm

def cosine_similarity(v1, v2):
    if norm(v1) == 0 or norm(v2) == 0:
        return 0.0
    return np.dot(v1, v2) / (norm(v1) * norm(v2))


In [41]:
jd = "Looking for a data analyst with experience in data cleaning and visualization"

services = [
    "AI-based data processing and analytics solutions",
    "Customer support outsourcing improves client satisfaction",
    "IT operations management and support"
]

# Pass a list of docs to compute_tfidf
all_docs = [jd] + services
tfidf_matrix, vocab = compute_tfidf(all_docs)

# Separate JD vector and services
jd_vector = tfidf_matrix[0]
service_vectors = tfidf_matrix[1:]

# Compute similarities
from numpy.linalg import norm
def cosine_similarity(v1, v2):
    if norm(v1) == 0 or norm(v2) == 0:
        return 0.0
    return np.dot(v1, v2) / (norm(v1) * norm(v2))

similarities = [cosine_similarity(jd_vector, sv) for sv in service_vectors]
best_match_index = int(np.argmax(similarities))

print("Similarity scores:", similarities)
print("Best matching service:", services[best_match_index])


Similarity scores: [0.19861931399202057, 0.0, 0.05154340996681279]
Best matching service: AI-based data processing and analytics solutions


In [75]:
import numpy as np
import math
from numpy.linalg import norm
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
import pickle

# ----------------- TF-IDF Utilities -----------------
class CustomTFIDF:
    def __init__(self):
        self.vocab = None
        self.idf = None

    def tokenizer(self, doc):
        return doc.lower().split()

    def build_vocab(self, docs):
        vocab = set()
        for doc in docs:
            vocab.update(self.tokenizer(doc))
        self.vocab = sorted(list(vocab))
        return self.vocab

    def compute_tf(self, doc):
        tokens = self.tokenizer(doc)
        tf = np.zeros(len(self.vocab))
        for i, term in enumerate(self.vocab):
            tf[i] = tokens.count(term) / len(tokens) if len(tokens) > 0 else 0
        return tf

    def compute_idf(self, docs):
        N = len(docs)
        idf = np.zeros(len(self.vocab))
        for i, term in enumerate(self.vocab):
            n_t = sum(1 for doc in docs if term in self.tokenizer(doc))
            idf[i] = math.log(N / (1 + n_t)) + 1
        self.idf = idf
        return idf

    def compute_tfidf(self, docs):
        self.build_vocab(docs)
        self.compute_idf(docs)
        tfidf_matrix = []
        for doc in docs:
            tf = self.compute_tf(doc)
            tfidf = tf * self.idf
            tfidf_matrix.append(tfidf)
        return np.array(tfidf_matrix)

    def cosine_similarity(self, v1, v2):
        if norm(v1) == 0 or norm(v2) == 0:
            return 0.0
        return np.dot(v1, v2) / (norm(v1) * norm(v2))

# ----------------- Mail Generator -----------------
class MailGenerator:
    def __init__(self, model_path, tokenizer_path, max_len=100, start_token='<start>', end_token='<end>'):
        self.model_path = model_path
        self.max_len = max_len
        self.start_token = start_token
        self.end_token = end_token

        # Load Keras model
        self.mail_model = load_model(model_path)

        # Load tokenizer.pkl
        with open(tokenizer_path, "rb") as f:
            self.tokenizer = pickle.load(f)

        # TF-IDF utility
        self.tfidf = CustomTFIDF()

    def generate_from_model(self, model_input, max_output_len=100):
        """
        Token-by-token generation for context-aware emails.
        """
        # Add start token to input
        model_input = f"{self.start_token} {model_input}"

        # Convert to sequence
        input_seq = self.tokenizer.texts_to_sequences([model_input])
        padded_input = pad_sequences(input_seq, maxlen=self.max_len, padding='post')

        output_seq = []
        current_input = padded_input

        for _ in range(max_output_len):
            preds = self.mail_model.predict(current_input, verbose=0)

            # Handle 2D output
            token_id = np.argmax(preds[0])
            word = self.tokenizer.index_word.get(token_id, '')

            # Stop if end token or unknown
            if word == self.end_token or word == '':
                break

            output_seq.append(token_id)

            # Append predicted token to current_input
            current_input = np.append(current_input, [[token_id]], axis=1)

        if output_seq:
            texts = self.tokenizer.sequences_to_texts([output_seq])
            return " ".join(texts)
        else:
            return ""

    def generate(self, jd, services):
        # Step 1: TF-IDF matching
        all_docs = [jd] + services
        tfidf_matrix = self.tfidf.compute_tfidf(all_docs)
        jd_vector = tfidf_matrix[0]
        service_vectors = tfidf_matrix[1:]
        similarities = [self.tfidf.cosine_similarity(jd_vector, sv) for sv in service_vectors]
        best_match_index = int(np.argmax(similarities))
        best_service = services[best_match_index]

        # Step 2: Prepare input exactly like training
        model_input = f"Job Description: {jd} Service: {best_service}"

        # Step 3: Generate context-aware email
        email = self.generate_from_model(model_input)
        return email, best_service, similarities

# ----------------- Usage Example -----------------
jd = "Looking for a data analyst with experience in data cleaning and visualization"
services = [
    "AI-based data processing and analytics solutions",
    "Customer support outsourcing improves client satisfaction",
    "IT operations management and support"
]

generator = MailGenerator(
    model_path="/Users/bimalthapa/Downloads/AI_Sales_Project/myenv/Final_Mail_Generator1_model.h5",
    tokenizer_path="/Users/bimalthapa/Downloads/AI_Sales_Project/myenv/tokenizer.pkl",
    max_len=100,
    start_token='<start>',
    end_token='<end>'
)

email, best_service, similarity_scores = generator.generate(jd, services)

print("Best Matching Service:", best_service)
print("Similarity Scores:", similarity_scores)
print("Generated Email:\n", email)


Best Matching Service: AI-based data processing and analytics solutions
Similarity Scores: [0.19861931399202057, 0.0, 0.05154340996681279]
Generated Email:
 vetted vetted vetted trust reaching well. i’m rahul desai, reaching out from talentgrid. i saw your recent job listing and wanted to reach out, and i’d be glad to assist you with the right talent or solution.

we help build agile teams quickly, minimizing hiring friction whether you're hiring short-term or scaling long-term, we’re confident we can help.

would a brief conversation this week work for you?

looking forward to your response.

best regards,
rahul menon
client recruitment account solutions
+1-202-555-1003
anna.li@talentgrid.co.uk
www.talentgrid.co.uk
 solutions
+1-202-555-1057
sophie.turner@devbridge.tech
www.devbridge.tech
 solutions
+1-202-555-1057
sophie.turner@devbridge.tech
www.devbridge.tech
 forward you're reaching out from swiftstack i saw your recent job listing and wanted to reach out, and i’d be glad to assis